# Model Baseline Evaluation

The following notebook proposes to study how the 3B parameters Mistral model behaves on the test set *before* any finetuning operation takes part.

In [1]:
from pathlib import Path
from datasets import load_from_disk
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import subprocess
from pathlib import Path
import transformers, sys
import shutil


### Loading the Datasets

In [2]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "sft"
MODEL_CACHE = PROJECT_ROOT / "models_cache"


In [3]:
test_ds = load_from_disk(dataset_path="../data/processed/test")
val_ds = load_from_disk(dataset_path="../data/processed/val")
train_ds = load_from_disk(dataset_path="../data/processed/train")

test_sft_ds = load_from_disk(dataset_path="../data/processed/sft/test_sft")
val_sft_ds = load_from_disk(dataset_path="../data/processed/sft/validation_sft")
train_sft_ds = load_from_disk(dataset_path="../data/processed/sft/train_sft")


### Loading the Model

In [4]:
print(transformers.__version__)
print(transformers.__file__)
print(sys.executable)


5.5.4
/home/riccorte/miniconda3/lib/python3.12/site-packages/transformers/__init__.py
/home/riccorte/miniconda3/bin/python


In [5]:
model_name = "Aratako/Ministral-3-3B-Instruct-2512-TextOnly"


In [6]:
OFFLOAD_DIR = PROJECT_ROOT / "offload"
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)


#### Run only If you want to download the model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=str(MODEL_CACHE),
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=str(MODEL_CACHE),
    torch_dtype="auto",
    device_map="auto",
    offload_folder=str(OFFLOAD_DIR),
)


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

#### Verifying that the download was succesful

In [ ]:
PROJECT_ROOT = Path("..")
MODEL_CACHE = PROJECT_ROOT / "models_cache"
OFFLOAD_DIR = PROJECT_ROOT / "offload"

print("MODEL CACHE EXISTS:", MODEL_CACHE.exists())
print("OFFLOAD EXISTS:", OFFLOAD_DIR.exists())

subprocess.run(["du", "-sh", str(MODEL_CACHE)])
subprocess.run(["du", "-sh", str(OFFLOAD_DIR)])


MODEL CACHE EXISTS: True
OFFLOAD EXISTS: True
3.7G	../models_cache
6.4G	../offload


CompletedProcess(args=['du', '-sh', '../offload'], returncode=0)

In [ ]:
files = []
for folder in [MODEL_CACHE, OFFLOAD_DIR]:
    if folder.exists():
        for path in folder.rglob("*"):
            if path.is_file():
                files.append((path.stat().st_size, path))

files = sorted(files, reverse=True)

for size, path in files[:20]:
    print(f"{size / 1024**2:.2f} MB  {path}")


3654.38 MB  ../models_cache/models--Aratako--Ministral-3-3B-Instruct-2512-TextOnly/snapshots/e5c347cdff40c5e53d23e6230b6504bfa18f6a3d/model.safetensors
3654.38 MB  ../models_cache/models--Aratako--Ministral-3-3B-Instruct-2512-TextOnly/blobs/47d7ef2f8bdeb57c653de1833a517f5d565e6f339db832a31fbeffc07b8c59e0
768.00 MB  ../offload/model.embed_tokens.weight.safetensors
54.00 MB  ../offload/model.layers.25.mlp.gate_proj.weight.safetensors
54.00 MB  ../offload/model.layers.25.mlp.down_proj.weight.safetensors
54.00 MB  ../offload/model.layers.24.mlp.gate_proj.weight.safetensors
54.00 MB  ../offload/model.layers.24.mlp.down_proj.weight.safetensors
54.00 MB  ../offload/model.layers.23.mlp.gate_proj.weight.safetensors
54.00 MB  ../offload/model.layers.23.mlp.down_proj.weight.safetensors
54.00 MB  ../offload/model.layers.22.mlp.gate_proj.weight.safetensors
54.00 MB  ../offload/model.layers.22.mlp.down_proj.weight.safetensors
54.00 MB  ../offload/model.layers.21.mlp.gate_proj.weight.safetensors
54.0

#### Run for each session to reload the offload cache

The following cells are used to clean the offload folder in case the previous session ended badly, allowig to reload the tokenizer and model

In [7]:
OFFLOAD_DIR = Path("../offload")

if OFFLOAD_DIR.exists():
    shutil.rmtree(OFFLOAD_DIR)

OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)
print("Recreated clean offload folder:", OFFLOAD_DIR)


Recreated clean offload folder: ../offload


In [8]:
model_name = "Aratako/Ministral-3-3B-Instruct-2512-TextOnly"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=str(MODEL_CACHE),
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float32,
)


`torch_dtype` is deprecated! Use `dtype` instead!
Using FP8 quantized models requires a GPU or XPU, we will default to dequantizing the model to bf16 since no GPU or XPU is available


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

: 

In [ ]:
print(type(model))
print(type(tokenizer))
print(model.device)


<class 'transformers.models.ministral3.modeling_ministral3.Ministral3ForCausalLM'>
<class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
meta


### Testing the Dataset

Checking the dataset

In [38]:
example = test_ds[302]
print(test_ds[0],"\n", test_ds.column_names, "\n",test_ds.shape)
example = test_ds[0]

print("Language code:", example["language_code"])
print("\nINPUT:\n", example["inputs"])
print("\nTARGET:\n", example["targets"])


{'inputs': 'How does globalization impact national economies?', 'targets': 'Globalization, the interconnectedness of economies worldwide, profoundly affects national economies. It facilitates the flow of goods, services, capital, and information across borders, fostering economic growth. However, it can also lead to income inequality and job displacement as industries may shift to regions with lower production costs. Additionally, globalization exposes economies to external shocks and financial crises. Countries engaging in global trade must implement policies that balance the benefits of globalization with the need to protect domestic industries and workers. Overall, effective management of globalization requires a strategic approach considering the opportunities and challenges it presents.\n\n', 'language': 'English', 'language_code': 'eng', 'annotation_type': 'original-annotations', 'user_id': '6a814b9adfea95c45ce3892528646106af167aad3bafa838279e891b9f979962'} 
 ['inputs', 'targets'

Creating the chat-style prompt to later do inference

In [ ]:
messages = [
    {"role": "user", "content": example["inputs"]}
]


apply the tokenizer chat template

In [ ]:
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

#print(prompt_text)


since the model is offloaded model.device is not useful so we verify the input device suitable

Now we tokenize the prompt and moved it into the input device

In [ ]:
inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
)


#### Generating an answer

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/home/riccorte/miniconda3/lib/python3.12/site-packages/transformers/generation/utils.py:2511: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on meta. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('meta') before running `.generate()`.
  warnings.warn(


RuntimeError: Tensor.item() cannot be called on meta tensors

The focus is only on the generated answer so we decode only the newly generated part

In [ ]:
prompt_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][prompt_len:]

prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("MODEL PREDICTION:\n")
print(prediction)


#### Compare target and prediction

In [ ]:
print("=== TARGET ===")
print(example["targets"])

print("\n=== PREDICTION ===")
print(prediction)


### Saving the Results